# Laboratório — Árvores de decisão: partições, impureza e estabilidade

Este laboratório reproduz os conceitos da Aula 09 com dados sintéticos. O teste é reservado antes da seleção e aberto uma única vez ao final.

**Hipóteses antes de executar**

1. uma árvore irrestrita terá score de treino maior, mas generalização inferior a uma árvore regularizada;
2. uma transformação positiva de escala não alterará as classes previstas;
3. reamostragens bootstrap alterarão o split raiz, revelando instabilidade estrutural.

Seed global: `20260908`. Não há download, credencial ou dependência de rede.

## Dependências

Ambiente recomendado:

```text
numpy>=1.26
pandas>=2.2
matplotlib>=3.8
scikit-learn>=1.4
```

No Colab, as bibliotecas já costumam estar disponíveis. Se necessário, descomente a instalação.

In [ ]:
# %pip install "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "scikit-learn>=1.4"
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

SEED = 20260908
np.random.seed(SEED)
warnings.filterwarnings("error")

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Gini e ganho calculados à mão

O nó pai tem 6 positivos e 4 negativos. O split cria uma folha pura com 4 positivos e outra com 2 positivos e 4 negativos.

In [ ]:
def gini(counts):
    counts = np.asarray(counts, dtype=float)
    probabilities = counts / counts.sum()
    return 1.0 - np.sum(probabilities**2)

gini_parent = gini([6, 4])
gini_left = gini([4, 0])
gini_right = gini([2, 4])
weighted_children = (4 / 10) * gini_left + (6 / 10) * gini_right
gain = gini_parent - weighted_children

assert np.isclose(gini_parent, 0.48)
assert np.isclose(weighted_children, 4 / 15)
assert np.isclose(gain, 0.21333333333333332)
print(f"Gini pai: {gini_parent:.6f}")
print(f"Impureza ponderada: {weighted_children:.6f}")
print(f"Ganho: {gain:.6f}")

## 2. Dados e teste lacrado

Cada linha representa uma unidade independente. As duas luas produzem uma fronteira não linear, adequada para visualizar partições alinhadas aos eixos. O teste recebe 25% dos dados e não participa das próximas decisões.

In [ ]:
X, y = make_moons(n_samples=1200, noise=0.28, random_state=SEED)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
feature_names = ["sinal_1", "sinal_2"]

assert X_dev.shape == (900, 2)
assert X_test.shape == (300, 2)
assert set(np.unique(y_dev)) == {0, 1}
print("Desenvolvimento:", X_dev.shape, "| teste reservado:", X_test.shape)
print("Proporção positiva no desenvolvimento:", f"{y_dev.mean():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_dev[:, 0], X_dev[:, 1], c=y_dev, cmap="coolwarm", s=18, alpha=0.75)
ax.set(xlabel="sinal_1", ylabel="sinal_2", title="Dados de desenvolvimento: duas classes não lineares")
ax.grid(alpha=0.2)
plt.show()

**Descrição do gráfico:** duas faixas curvas intercaladas representam as classes. Uma única reta não separa perfeitamente as observações; uma árvore aproxima a fronteira com vários cortes horizontais e verticais.

## 3. Baseline e folds fixos

Usaremos acurácia balanceada para manter o protocolo explícito. As classes são equilibradas, então ela se aproxima da acurácia comum. Todos os candidatos recebem exatamente os mesmos folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
baseline = DummyClassifier(strategy="prior")
baseline_scores = cross_validate(
    baseline, X_dev, y_dev, cv=cv, scoring="balanced_accuracy", return_train_score=True
)
baseline_cv = baseline_scores["test_score"].mean()
assert 0.49 <= baseline_cv <= 0.51
print(f"Baseline CV: {baseline_cv:.6f}")

## 4. Caminho de profundidade

Primeiro variamos somente `max_depth`, mantendo `min_samples_leaf=1`. O contraste treino–validação evidencia a troca entre viés e variância.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 12, None]
rows = []
for depth in depths:
    estimator = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    scores = cross_validate(
        estimator, X_dev, y_dev, cv=cv, scoring="balanced_accuracy", return_train_score=True
    )
    estimator.fit(X_dev, y_dev)
    rows.append({
        "max_depth": "None" if depth is None else str(depth),
        "train_mean": scores["train_score"].mean(),
        "cv_mean": scores["test_score"].mean(),
        "cv_std": scores["test_score"].std(ddof=1),
        "leaves": estimator.get_n_leaves(),
    })

depth_results = pd.DataFrame(rows)
assert depth_results.loc[depth_results.max_depth == "None", "train_mean"].iloc[0] > 0.99
depth_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(depth_results))
ax.plot(x, depth_results.train_mean, "o-", label="treino")
ax.plot(x, depth_results.cv_mean, "o-", label="validação cruzada")
ax.fill_between(
    x,
    depth_results.cv_mean - depth_results.cv_std,
    depth_results.cv_mean + depth_results.cv_std,
    alpha=0.18,
)
ax.set_xticks(x, depth_results.max_depth)
ax.set(xlabel="max_depth", ylabel="acurácia balanceada", title="Profundidade: ajuste versus generalização")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

**Descrição do gráfico:** o score de treino cresce com a profundidade e chega perto de 1 na árvore irrestrita. A validação melhora inicialmente e depois satura ou piora, formando um vão de generalização.

## 5. Seleção de regularização apenas no desenvolvimento

Agora combinamos profundidade, tamanho mínimo de folha e `ccp_alpha`. A grade é definida antes da avaliação e o teste continua lacrado.

In [ ]:
param_grid = {
    "max_depth": [3, 4, 5, 6, None],
    "min_samples_leaf": [1, 5, 15],
    "ccp_alpha": [0.0, 0.001, 0.003, 0.01],
}
search = GridSearchCV(
    DecisionTreeClassifier(random_state=SEED),
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
search.fit(X_dev, y_dev)

best_cv = search.best_score_
best_params = search.best_params_
assert best_cv > baseline_cv + 0.25
print("Melhores parâmetros:", best_params)
print(f"Melhor CV: {best_cv:.6f}")

A melhor combinação não é uma lei universal; ela responde a estes dados, folds, métrica e grade. Em trabalho rigoroso, registre também candidatos descartados e incerteza entre folds.

In [ ]:
cv_table = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cols = ["param_max_depth", "param_min_samples_leaf", "param_ccp_alpha",
        "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]
cv_table[cols].head(10).round(5)

## 6. Estrutura final antes do teste

`best_estimator_` já foi reajustado em todo o conjunto de desenvolvimento. Exportamos suas regras, profundidade e número de folhas sem usar o teste.

In [ ]:
selected_tree = search.best_estimator_
rules = export_text(selected_tree, feature_names=feature_names, decimals=3)
print("Profundidade efetiva:", selected_tree.get_depth())
print("Número de folhas:", selected_tree.get_n_leaves())
print(rules)
assert selected_tree.get_n_leaves() >= 2

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_tree(
    selected_tree,
    feature_names=feature_names,
    class_names=["classe 0", "classe 1"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
ax.set_title("Primeiros níveis da árvore selecionada")
plt.show()

**Descrição do gráfico:** os primeiros níveis exibem feature, limiar, Gini, número de amostras e distribuição de classes. A cor fica mais intensa quando uma classe domina o nó. A figura é truncada no terceiro nível para permanecer legível.

## 7. Fronteira em degraus

A visualização revela a natureza constante por regiões. Pontos próximos podem receber decisões diferentes quando estão em lados opostos de um limiar.

In [ ]:
x0_min, x0_max = X_dev[:, 0].min() - 0.3, X_dev[:, 0].max() + 0.3
x1_min, x1_max = X_dev[:, 1].min() - 0.3, X_dev[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 350), np.linspace(x1_min, x1_max, 350))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = selected_tree.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
ax.scatter(X_dev[:, 0], X_dev[:, 1], c=y_dev, cmap="coolwarm", s=14, alpha=0.7)
ax.set(xlabel="sinal_1", ylabel="sinal_2", title="Fronteira de decisão da árvore selecionada")
ax.grid(alpha=0.2)
plt.show()

**Descrição do gráfico:** a fronteira curva é aproximada por blocos retangulares. O contorno em degraus decorre das regras univariadas alinhadas aos eixos.

## 8. Caminho de custo-complexidade

O caminho mostra os valores de `ccp_alpha` em que alguma subárvore é removida. Ele serve para compreender a poda; a escolha final já foi feita por validação cruzada.

In [ ]:
unpruned = DecisionTreeClassifier(random_state=SEED).fit(X_dev, y_dev)
path = unpruned.cost_complexity_pruning_path(X_dev, y_dev)
alphas, impurities = path.ccp_alphas, path.impurities

assert np.all(np.diff(alphas) >= -1e-15)
assert np.all(np.diff(impurities) >= -1e-15)
fig, ax = plt.subplots(figsize=(7, 4))
ax.step(alphas, impurities, where="post")
ax.set(xlabel="ccp_alpha efetivo", ylabel="impureza total das folhas", title="Caminho de poda por custo-complexidade")
ax.grid(alpha=0.25)
plt.show()
print("Nós sem poda:", unpruned.tree_.node_count, "| folhas:", unpruned.get_n_leaves())

**Descrição do gráfico:** conforme `ccp_alpha` aumenta, a impureza tolerada cresce porque folhas são removidas. A sequência é monotônica, indo da árvore grande até a raiz.

## 9. Escala: ordem preservada

Multiplicar a primeira feature por 1.000 não muda sua ordem. Ajustamos duas árvores idênticas em uma divisão interna do desenvolvimento e comparamos previsões, sem tocar no teste.

In [ ]:
X_subtrain, X_check, y_subtrain, y_check = train_test_split(
    X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=SEED + 1
)
params = {**best_params, "random_state": SEED}
tree_original = DecisionTreeClassifier(**params).fit(X_subtrain, y_subtrain)

scale = np.array([1000.0, 1.0])
tree_scaled = DecisionTreeClassifier(**params).fit(X_subtrain * scale, y_subtrain)
pred_original = tree_original.predict(X_check)
pred_scaled = tree_scaled.predict(X_check * scale)
agreement = np.mean(pred_original == pred_scaled)

assert agreement == 1.0
print(f"Concordância após mudança positiva de escala: {agreement:.6f}")

A igualdade não autoriza remover o pipeline: imputação, encoding, seleção e qualquer transformação aprendida ainda devem respeitar os folds.

## 10. Estabilidade do split raiz por bootstrap

Treinamos 100 árvores com a mesma configuração em reamostragens do desenvolvimento. Medimos qual feature foi escolhida na raiz e a variação do limiar na escala original.

In [ ]:
rng = np.random.default_rng(SEED)
root_features = []
root_thresholds = []
for _ in range(100):
    idx = rng.integers(0, len(X_dev), size=len(X_dev))
    model = DecisionTreeClassifier(**params).fit(X_dev[idx], y_dev[idx])
    root_features.append(model.tree_.feature[0])
    root_thresholds.append(model.tree_.threshold[0])

root_features = np.asarray(root_features)
root_thresholds = np.asarray(root_thresholds)
counts = pd.Series(root_features).map(dict(enumerate(feature_names))).value_counts()
threshold_sd = root_thresholds.std(ddof=1)

assert counts.sum() == 100
assert threshold_sd > 0
print("Frequência da feature na raiz:")
print(counts.to_string())
print(f"Desvio-padrão dos limiares da raiz: {threshold_sd:.6f}")

A seed torna o experimento repetível; a diversidade entre bootstraps mede outra propriedade: sensibilidade aos dados. Uma explicação baseada em um único split deve declarar essa instabilidade.

## 11. Avaliação final única

Somente agora calculamos as métricas no teste reservado. Não voltaremos à grade após observar estes números.

In [ ]:
test_pred = selected_tree.predict(X_test)
test_proba = selected_tree.predict_proba(X_test)
test_bal_acc = balanced_accuracy_score(y_test, test_pred)
test_log_loss = log_loss(y_test, test_proba, labels=[0, 1])

assert test_bal_acc > baseline_cv + 0.25
assert np.isfinite(test_log_loss)
print(f"Acurácia balanceada no teste: {test_bal_acc:.6f}")
print(f"Log-loss no teste: {test_log_loss:.6f}")
print(f"Ganho sobre baseline: {test_bal_acc - baseline_cv:.6f}")

## Conclusão auditável

- A árvore representou a fronteira não linear com regras em degraus.
- A curva de profundidade mostrou que ajuste de treino e generalização não são sinônimos.
- A configuração foi selecionada exclusivamente no desenvolvimento.
- A transformação positiva de escala preservou todas as classes previstas no conjunto interno.
- O bootstrap revelou variação da regra raiz, mesmo com configuração fixa.
- O teste foi utilizado uma única vez.

**O que o experimento não sustenta:** causalidade dos splits, superioridade universal da grade escolhida, calibração perfeita das probabilidades ou estabilidade em outro domínio. Na próxima aula, a agregação de árvores atacará diretamente parte da variância observada.